In [ ]:
%pip install -q kafka-python-ng websocket-client

In [ ]:
%%writefile /workspace/binance_producer.py
import json
import websocket
from kafka import KafkaProducer

KAFKA_BROKER = "crypto-kafka:9092"
TOPIC = "crypto-trades"

producer = KafkaProducer(
    bootstrap_servers=[KAFKA_BROKER],
    key_serializer=lambda key: key.encode("utf-8"),
    value_serializer=lambda value: json.dumps(value).encode("utf-8")
)

def on_message(ws, message):
    try:
        payload = json.loads(message)
        data = payload.get("data")
        if not data:
            return

        trade_event = {
            "event_time": data.get("E"),
            "symbol": data.get("s"),
            "price": float(data.get("p")),
            "quantity": float(data.get("q")),
            "is_buyer_maker": data.get("m")
        }

        producer.send(
            TOPIC,
            key=trade_event["symbol"],
            value=trade_event
        )
        print(
            "Sent: {} | Price: {} | Qty: {}".format(
                trade_event["symbol"],
                trade_event["price"],
                trade_event["quantity"]
            )
        )
    except Exception as error:
        print("Error processing message: {}".format(error))

def on_error(ws, error):
    print("WebSocket error: {}".format(error))

def on_close(ws, close_status_code, close_msg):
    print("WebSocket closed")
    producer.close()

def on_open(ws):
    print("Connected to Binance WebSocket")

if __name__ == "__main__":
    streams = "btcusdt@trade/ethusdt@trade/solusdt@trade/bnbusdt@trade/xrpusdt@trade"
    ws_url = "wss://stream.binance.com:9443/stream?streams={}".format(streams)

    ws = websocket.WebSocketApp(
        ws_url,
        on_open=on_open,
        on_message=on_message,
        on_error=on_error,
        on_close=on_close
    )
    ws.run_forever()

Spark Streaming + HDFS

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, sum as spark_sum, window, from_unixtime
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, BooleanType

# 1. Initializing Spark with the project cluster
spark = SparkSession.builder \
    .master("spark://crypto-spark-master:7077") \
    .appName("CryptoLivePipeline") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

# 2. JSON schema sent from Binance
schema = StructType([
    StructField("event_time", LongType(), True),
    StructField("symbol", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("quantity", DoubleType(), True),
    StructField("is_buyer_maker", BooleanType(), True)
])

# 3. Reading the flow from Kafka
raw_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "crypto-kafka:9092") \
    .option("subscribe", "crypto-trades") \
    .option("startingOffsets", "latest") \
    .load()

# 4. Parsing JSON and converting Unix milliseconds into regular timestamps
parsed_stream = raw_stream \
    .selectExpr("CAST(value AS STRING) as json_str") \
    .select(from_json(col("json_str"), schema).alias("data")) \
    .select("data.*") \
    .withColumn("timestamp", from_unixtime(col("event_time") / 1000).cast("timestamp")) \
    .withColumn("trade_value", col("price") * col("quantity"))


# Stream 1: Calculate VWAP and volume in one-minute windows

vwap_stats = parsed_stream \
    .withWatermark("timestamp", "1 minute") \
    .groupBy(window(col("timestamp"), "1 minute"), col("symbol")) \
    .agg(
        (spark_sum("trade_value") / spark_sum("quantity")).alias("vwap"),
        spark_sum("quantity").alias("total_crypto_traded")
    )

query_dashboard = vwap_stats.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("vwap_dashboard") \
    .start()


# Stream 2: Write raw trade records to HDFS in Parquet format

query_hdfs = parsed_stream.writeStream \
    .outputMode("append") \
    .format("parquet") \
    .option("path", "hdfs://crypto-namenode:9000/data/raw/crypto-trades/") \
    .option("checkpointLocation", "hdfs://crypto-namenode:9000/checkpoints/crypto-trades/") \
    .start()

print("Streaming pipeline started")

Live dashboard

In [ ]:
import time
from IPython.display import clear_output

try:
    for i in range(12):
        time.sleep(5)
        clear_output(wait=True)
        print(f"=== Live Binance VWAP & Volume (Update {i+1}/12) ===")
        spark.sql("""
            SELECT 
                date_format(window.start, 'HH:mm:ss') AS window_start,
                symbol, 
                ROUND(vwap, 2) AS vwap_usd,
                ROUND(total_crypto_traded, 2) AS total_volume_coins
            FROM vwap_dashboard
            ORDER BY window.start DESC, vwap_usd DESC
        """).show(truncate=False)
except KeyboardInterrupt:
    print("Dashboard view stopped.")

Stopping streams and checking the Data Lake (HDFS)

In [ ]:
# 1. Stopping all active threads
for q in spark.streams.active:
    q.stop()
print("Streaming jobs stopped.")

# 2. Reading historical data, that were recorded in HDFS
print("Loading historical Parquet data from HDFS...")
historical_df = spark.read.parquet("hdfs://crypto-namenode:9000/data/raw/crypto-trades/")

# 3. Printing statistics for the stored Data Lake
total_records = historical_df.count()
print(f"Trade records in HDFS: {total_records}")

print("Sample of raw data stored in Parquet columnar format:")
historical_df.select("timestamp", "symbol", "price", "quantity").show(5, truncate=False)

Launching a Spark benchmark and preparing data for Hadoop

In [ ]:
import time

print("1. Spark SQL benchmark")
start_time = time.time()

# Spark performs aggregation using the Catalyst optimizer and in-memory processing
spark.sql("""
    SELECT symbol, ROUND(SUM(quantity), 2) AS total_volume
    FROM parquet.`hdfs://crypto-namenode:9000/data/raw/crypto-trades/`
    GROUP BY symbol
    ORDER BY total_volume DESC
""").show()

spark_time = time.time() - start_time
print(f"-> Apache Spark Execution Time: {spark_time:.4f} seconds\n")

print("2. Preparing CSV data for Hadoop MapReduce")
# Saving a text copy of the data for classic MapReduce.
historical_df.select("symbol", "quantity") \
    .write.mode("overwrite") \
    .csv("hdfs://crypto-namenode:9000/data/raw/crypto-trades-csv/")
print("CSV snapshot saved to HDFS")

Saving scripts for Hadoop

In [ ]:
%%writefile /workspace/mapper.py
import sys
for line in sys.stdin:
    parts = line.strip().split(',')
    if len(parts) == 2:
        print("{}\t{}".format(parts[0], parts[1]))

In [ ]:
%%writefile /workspace/reducer.py
import sys
current_symbol = None
total_qty = 0.0

for line in sys.stdin:
    try:
        symbol, qty = line.strip().split('\t')
        qty = float(qty)
    except ValueError:
        continue
        
    if current_symbol == symbol:
        total_qty += qty
    else:
        if current_symbol:
            print("{}\t{}".format(current_symbol, round(total_qty, 2)))
        current_symbol = symbol
        total_qty = qty
        
if current_symbol:
    print("{}\t{}".format(current_symbol, round(total_qty, 2)))